# Extract logistic regression model coefficients per CellProfiler feature

The coefficients from the final model will either be positive (x > 0), where if the feature value increases, the more likely the cell is vehicle treated (non-failing), negative (x < 0), where if the feature value increases, the more likely the cell is TGF-β treated (fibrotic), or zero (x = 0), where that feature has no impact on the prediction.

## Import libraries

In [1]:
import pathlib

import pandas as pd
from joblib import load

## Set paths and variables

In [2]:
# Name of the model to extract coefficients from
model_name = "model_TGFB_vs_Veh"

# Path to the final model
path_to_final_model = pathlib.Path(
    f"./models/{model_name}_final_downsample.joblib"
).resolve(strict=True)

# Path to the label encoder to confirm which class is positive
path_to_encoder = pathlib.Path(
    "./encoder_results/label_encoder_global.joblib"
).resolve(strict=True)

# Output dir for coefficients data
coeff_dir = pathlib.Path("./coeff_data")
coeff_dir.mkdir(exist_ok=True)

# Channels in this dataset (used to find the channel in each feature name)
channels = ["DNA", "ER", "Golgi", "Mito", "Actin"]

## Load in the final model and label encoder

In [3]:
# Load in final model and label encoder
final_model = load(path_to_final_model)
label_encoder = load(path_to_encoder)

# Confirm the positive class (positive coefficients) is the vehicle class
assert list(label_encoder.classes_) == [
    "TGFB",
    "Veh",
], f"Unexpected classes: {label_encoder.classes_}"
print(
    f"Negative coefficients predict {label_encoder.classes_[0]}, positive coefficients predict {label_encoder.classes_[1]}"
)

Negative coefficients predict TGFB, positive coefficients predict Veh


## Collect coefficients from the model and match with the correct feature in a dataframe

The feature names are stored in the model, so we use them directly to match each coefficient to its feature.
We also split out the compartment, feature group, and channel from each feature name for visualization.
The channel is the first channel name found in the feature name (for correlation features, this is the first of the two channels), and features without a channel (e.g., AreaShape and Neighbors) are labelled as `other`.

In [4]:
# Get the coefficients and feature names from the model
coefficients = final_model.coef_.flatten()
feature_columns = final_model.feature_names_in_

print(f"Number of coefficients: {len(coefficients)}")
assert len(coefficients) == len(
    feature_columns
), "The number of coefficients does not match the number of features"


def find_channel(feature: str) -> str:
    """Find the first channel in a feature name.

    Args:
        feature (str): CellProfiler feature name

    Returns:
        str: the first channel found in the feature name, or "other" if there is no channel
    """
    for part in feature.split("_")[2:]:
        if part in channels:
            return part
    return "other"


# Create a DataFrame with the coefficients and features
coefficients_df = pd.DataFrame({"Feature": feature_columns, "Coefficient": coefficients})

# Split out components of the feature name for visualization
coefficients_df["Compartment"] = coefficients_df["Feature"].str.split("_").str[0]
coefficients_df["Feature_Group"] = coefficients_df["Feature"].str.split("_").str[1]
coefficients_df["Channel"] = coefficients_df["Feature"].apply(find_channel)

# Save the all coefficient data into a CSV file prior to splitting
coefficients_df.to_csv(coeff_dir / "all_coeffs.csv", index=False)

print(coefficients_df["Channel"].value_counts())
coefficients_df.head()

Number of coefficients: 1007
Channel
ER       200
DNA      192
Golgi    180
Actin    179
Mito     163
other     93
Name: count, dtype: int64


,Feature,Coefficient,Compartment,Feature_Group,Channel
0,Cytoplasm_AreaShape_Compactness,-0.012132,Cytoplasm,AreaShape,other
1,Cytoplasm_AreaShape_Eccentricity,0.030925,Cytoplasm,AreaShape,other
2,Cytoplasm_AreaShape_FormFactor,0.047397,Cytoplasm,AreaShape,other
3,Cytoplasm_AreaShape_MajorAxisLength,0.022915,Cytoplasm,AreaShape,other
4,Cytoplasm_AreaShape_Zernike_2_2,0.021244,Cytoplasm,AreaShape,other


## Split the data frame by negative, positive, and zero coefficients which relate to different class importance

In [5]:
# Split into DataFrames with positive, negative, and zero coefficients
veh_coeffs_df = coefficients_df[coefficients_df["Coefficient"] > 0].copy()
tgfb_coeffs_df = coefficients_df[coefficients_df["Coefficient"] < 0].copy()
zero_coeffs_df = coefficients_df[coefficients_df["Coefficient"] == 0].copy()

# Make the values in tgfb_coeffs_df positive to prevent issues during plotting
tgfb_coeffs_df["Coefficient"] = tgfb_coeffs_df["Coefficient"].abs()

# Rename the coefficient columns
veh_coeffs_df = veh_coeffs_df.rename(columns={"Coefficient": "Veh_Coeffs"})
tgfb_coeffs_df = tgfb_coeffs_df.rename(columns={"Coefficient": "TGFB_Coeffs"})
zero_coeffs_df = zero_coeffs_df.rename(columns={"Coefficient": "Zero_Coeffs"})

# Save the coefficient data into the "coeff_data" folder
veh_coeffs_df.to_csv(coeff_dir / "veh_coeffs.csv", index=False)
tgfb_coeffs_df.to_csv(coeff_dir / "tgfb_coeffs.csv", index=False)
zero_coeffs_df.to_csv(coeff_dir / "zero_coeffs.csv", index=False)

print("Positive (Veh) coefficients:", veh_coeffs_df.shape[0])
print("Negative (TGFB) coefficients:", tgfb_coeffs_df.shape[0])
print("Zero coefficients:", zero_coeffs_df.shape[0])

Positive (Veh) coefficients: 472
Negative (TGFB) coefficients: 509
Zero coefficients: 26


## Explore the coefficients

In [6]:
# Show the top ten ranking features for predicting the vehicle (Veh) class
coefficients_df.sort_values(by="Coefficient", ascending=False).head(10)

,Feature,Coefficient,Compartment,Feature_Group,Channel
708,Nuclei_Intensity_MinIntensityEdge_DNA,0.202812,Nuclei,Intensity,DNA
265,Cytoplasm_Texture_AngularSecondMoment_Actin_3_...,0.195382,Cytoplasm,Texture,Actin
25,Cytoplasm_Correlation_Correlation_Actin_Mito,0.164496,Cytoplasm,Correlation,Actin
127,Cytoplasm_RadialDistribution_RadialCV_Actin_4of4,0.162257,Cytoplasm,RadialDistribution,Actin
653,Nuclei_Correlation_K_Actin_DNA,0.157353,Nuclei,Correlation,Actin
655,Nuclei_Correlation_K_Actin_Golgi,0.156373,Nuclei,Correlation,Actin
305,Cytoplasm_Texture_DifferenceVariance_Actin_3_0...,0.148623,Cytoplasm,Texture,Actin
51,Cytoplasm_Correlation_RWC_Actin_ER,0.145295,Cytoplasm,Correlation,Actin
921,Nuclei_Texture_AngularSecondMoment_Actin_3_01_256,0.143012,Nuclei,Texture,Actin
423,Cells_Intensity_MinIntensityEdge_DNA,0.142102,Cells,Intensity,DNA


In [7]:
# Show the top ten ranking features for predicting the TGF-β (TGFB) class
coefficients_df.sort_values(by="Coefficient", ascending=True).head(10)

,Feature,Coefficient,Compartment,Feature_Group,Channel
422,Cells_Intensity_MinIntensityEdge_Actin,-0.405021,Cells,Intensity,Actin
69,Cytoplasm_Granularity_1_Actin,-0.319056,Cytoplasm,Granularity,Actin
680,Nuclei_Granularity_1_Actin,-0.259633,Nuclei,Granularity,Actin
79,Cytoplasm_Intensity_LowerQuartileIntensity_Actin,-0.236342,Cytoplasm,Intensity,Actin
416,Cells_Intensity_MeanIntensityEdge_Actin,-0.231649,Cells,Intensity,Actin
77,Cytoplasm_Intensity_IntegratedIntensityEdge_Actin,-0.212890,Cytoplasm,Intensity,Actin
99,Cytoplasm_Intensity_MinIntensityEdge_Golgi,-0.202920,Cytoplasm,Intensity,Golgi
424,Cells_Intensity_MinIntensityEdge_Mito,-0.180875,Cells,Intensity,Mito
712,Nuclei_Intensity_MinIntensity_Actin,-0.170266,Nuclei,Intensity,Actin
666,Nuclei_Correlation_K_Mito_Actin,-0.148749,Nuclei,Correlation,Mito


## Add ranking column with sorted descending values and save the CSV for visualization

Rank is based on the highest positive coefficient, which will have rank one, and then descending from there.

In [8]:
# Sort coefficients_df by descending order
ranked_coeffs_df = coefficients_df.sort_values(by="Coefficient", ascending=False).copy()

# Add a new column 'Rank'
ranked_coeffs_df["Rank"] = range(1, len(ranked_coeffs_df) + 1)

# Save the ranked df
ranked_coeffs_df.to_csv(coeff_dir / "ranked_coeffs.csv", index=False)

print(ranked_coeffs_df.shape)
ranked_coeffs_df.head()

(1007, 6)


,Feature,Coefficient,Compartment,Feature_Group,Channel,Rank
708,Nuclei_Intensity_MinIntensityEdge_DNA,0.202812,Nuclei,Intensity,DNA,1
265,Cytoplasm_Texture_AngularSecondMoment_Actin_3_...,0.195382,Cytoplasm,Texture,Actin,2
25,Cytoplasm_Correlation_Correlation_Actin_Mito,0.164496,Cytoplasm,Correlation,Actin,3
127,Cytoplasm_RadialDistribution_RadialCV_Actin_4of4,0.162257,Cytoplasm,RadialDistribution,Actin,4
653,Nuclei_Correlation_K_Actin_DNA,0.157353,Nuclei,Correlation,Actin,5
